In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import anndata
from scipy.stats import median_abs_deviation
import scrublet as scr

## Read in data and make names unique

In [2]:
ad_path = 'filtered_feature_bc_matrix02.h5'

adata = sc.read_10x_h5(ad_path)

/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [3]:
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 5114 × 36601
    var: 'gene_ids', 'feature_types', 'genome'

In [4]:
sample = []
samples = ['SoftMi', 'SoftNon', 'StiffMi', 'StiffNon']

for umi in adata.obs_names:
    sample.append(samples[int(umi[-1])-1])

sample

adata.obs['sample'] = sample

# Quality Control

### Filter low quality reads

In [5]:
# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes.
adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))

In [6]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)
adata

AnnData object with n_obs × n_vars = 5114 × 36601
    obs: 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'

In [7]:

p2 = sc.pl.violin(adata, "pct_counts_mt", groupby='sample')
#p3 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  ax = sns.violinplot(


In [8]:
sc.pl.violin(adata, "total_counts", groupby='sample', log=True, cut=0)
plt.show()

/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  ax = sns.violinplot(


In [10]:
def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

In [11]:
adata.obs["outlier"] = (
    is_outlier(adata, "log1p_total_counts", 5)
    | is_outlier(adata, "log1p_n_genes_by_counts", 5)
    | is_outlier(adata, "pct_counts_in_top_20_genes", 5)
)
adata.obs.outlier.value_counts()

outlier
False    4112
True     1002
Name: count, dtype: int64

### Filter out low gene and cell counts

In [12]:
mad_cutoff=3

In [13]:
print('Number of genes before filtering: {:d}'.format(adata.n_vars))
print('Number of cells before filtering: {:d}'.format(adata.n_obs))

print('Removing', np.sum(is_outlier(adata, "log1p_total_counts", mad_cutoff) | is_outlier(adata, "log1p_n_genes_by_counts", mad_cutoff)), 'cells that don\'t meet MAD')
adata = adata[~(is_outlier(adata, "log1p_total_counts", mad_cutoff) | is_outlier(adata, "log1p_n_genes_by_counts", mad_cutoff))]

print("Removing", sum(adata.obs['pct_counts_mt'] > 20), "cells with mt-frac > 20%")
adata = adata[adata.obs['pct_counts_mt'] < 20]

#min 20 cells- filters out 0 count genes
sc.pp.filter_genes(adata, min_cells=20)

print('Number of genes after filtering: {:d}'.format(adata.n_vars))
print('Number of cells after filtering: {:d}'.format(adata.n_obs))

Number of genes before filtering: 36601
Number of cells before filtering: 5114
Removing 1061 cells that don't meet MAD
Removing 203 cells with mt-frac > 20%


/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:250: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var['n_cells'] = number


Number of genes after filtering: 15404
Number of cells after filtering: 3850


### Filter out cells with high mitrochrondrial dna (>20% mt)

In [15]:
p2 = sc.pl.violin(adata, "pct_counts_mt", groupby='sample')
#p3 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
sc.pl.violin(adata, "total_counts", groupby='sample', log=True, cut=0)
plt.show()

/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  ax = sns.violinplot(
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 


### Doublet Detection

In [17]:
# Given a raw (unnormalized) UMI counts matrix counts_matrix with cells as rows and genes as columns, calculate a doublet score for each cell:

scrub = scr.Scrublet(adata.X, expected_doublet_rate=0.023)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.31
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 2.7%
Overall doublet rate:
	Expected   = 2.3%
	Estimated  = 3.8%
Elapsed time: 11.8 seconds


In [18]:
scrub.plot_histogram()

(<Figure size 800x300 with 2 Axes>,
 array([<Axes: title={'center': 'Observed transcriptomes'}, xlabel='Doublet score', ylabel='Prob. density'>,
        <Axes: title={'center': 'Simulated doublets'}, xlabel='Doublet score', ylabel='Prob. density'>],
       dtype=object))

In [20]:
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublets"] = predicted_doublets
adata.obs.predicted_doublets.value_counts()

predicted_doublets
False    3846
True        4
Name: count, dtype: int64

# Assign barcodes


In [ ]:
assignment_dict = pd.read_table('barcodeSeq02/lineage_assignment.txt', delimiter='\t', index_col=0, header=None)

In [23]:
assignment_dict = assignment_dict.to_dict()[1]

In [24]:
lineage_obs = []

for cell in adata.obs_names:
    if cell[:-2] in assignment_dict.keys():
        lineage_obs.append(assignment_dict[cell[:-2]])
    else:
        lineage_obs.append('none')

adata.obs['lineage'] = lineage_obs

## Normalize 

In [ ]:
adata.var_names_make_unique()

In [ ]:
adata.layers['raw_counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e6)

In [ ]:
sc.pp.log1p(adata)
adata.X.toarray()

In [ ]:
adata.layers['norm_counts'] = adata.X.copy()

In [ ]:
adata.layers['cc_regressed'] = adata.X.copy()

In [ ]:
anndata.AnnData.write(adata, 'HK177_02_processed.h5ad')